In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, BatchNormalization
from keras.regularizers import l1_l2
from keras.optimizers import Adam, RMSprop, SGD
from keras.callbacks import EarlyStopping

import optuna

In [2]:
df = pd.read_csv("medical_insurance.csv")

In [3]:
df.shape

(100000, 54)

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.isnull().sum()

person_id                          0
age                                0
sex                                0
region                             0
urban_rural                        0
income                             0
education                          0
marital_status                     0
employment_status                  0
household_size                     0
dependents                         0
bmi                                0
smoker                             0
alcohol_freq                   30083
visits_last_year                   0
hospitalizations_last_3yrs         0
days_hospitalized_last_3yrs        0
medication_count                   0
systolic_bp                        0
diastolic_bp                       0
ldl                                0
hba1c                              0
plan_type                          0
network_tier                       0
deductible                         0
copay                              0
policy_term_years                  0
p

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 54 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   person_id                    100000 non-null  int64  
 1   age                          100000 non-null  int64  
 2   sex                          100000 non-null  object 
 3   region                       100000 non-null  object 
 4   urban_rural                  100000 non-null  object 
 5   income                       100000 non-null  float64
 6   education                    100000 non-null  object 
 7   marital_status               100000 non-null  object 
 8   employment_status            100000 non-null  object 
 9   household_size               100000 non-null  int64  
 10  dependents                   100000 non-null  int64  
 11  bmi                          100000 non-null  float64
 12  smoker                       100000 non-null  object 
 13  

In [7]:
df["alcohol_freq"] = df["alcohol_freq"].fillna(df["alcohol_freq"].mode()[0])

In [8]:
df.isnull().sum()

person_id                      0
age                            0
sex                            0
region                         0
urban_rural                    0
income                         0
education                      0
marital_status                 0
employment_status              0
household_size                 0
dependents                     0
bmi                            0
smoker                         0
alcohol_freq                   0
visits_last_year               0
hospitalizations_last_3yrs     0
days_hospitalized_last_3yrs    0
medication_count               0
systolic_bp                    0
diastolic_bp                   0
ldl                            0
hba1c                          0
plan_type                      0
network_tier                   0
deductible                     0
copay                          0
policy_term_years              0
policy_changes_last_2yrs       0
provider_quality               0
risk_score                     0
annual_med

In [9]:
X = df.drop(
    columns=[
        "person_id",
        "annual_medical_cost",
        "annual_premium",
        "monthly_premium"
    ]
)

y = df["annual_medical_cost"]

In [10]:
X_train_full , X_test,y_train_full,y_test = train_test_split(X,y,test_size=0.2,random_state=21)

In [11]:
X_train,X_val,y_train,y_val = train_test_split(X_train_full,y_train_full,test_size=0.2,random_state=21)

In [12]:
num_cols = X.select_dtypes(exclude = "object").columns
cat_cols = X.select_dtypes(include = "object").columns

In [14]:
preprocessor = ColumnTransformer([("Scaling",StandardScaler(),num_cols),
                                  ("Encoding",OneHotEncoder(drop = "first",handle_unknown="ignore"),cat_cols)],
                                 remainder = "drop")

In [15]:
X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

In [16]:
model = Sequential([Input(shape=(X_train_t.shape[1],)),
                    Dense(32,activation="relu"),
                    Dense(64,activation="relu"),
                    Dense(1,activation="linear")])

In [17]:
model.compile(optimizer="Adam",loss = "mse",metrics = ["r2_score"])

In [18]:
early = EarlyStopping(monitor = "val_loss",patience = 5 , restore_best_weights=True)

In [19]:
model.fit(X_train_t,y_train,
          epochs = 50,
          validation_data=(X_val_t,y_val),
          batch_size=256,
          callbacks=[early],
          verbose = 2)

Epoch 1/50
250/250 - 6s - 23ms/step - loss: 16748952.0000 - r2_score: -7.2187e-01 - val_loss: 11683708.0000 - val_r2_score: -1.6872e-01
Epoch 2/50
250/250 - 2s - 7ms/step - loss: 7531402.0000 - r2_score: 0.2257 - val_loss: 6322360.5000 - val_r2_score: 0.3676
Epoch 3/50
250/250 - 2s - 6ms/step - loss: 5588511.5000 - r2_score: 0.4255 - val_loss: 5278083.0000 - val_r2_score: 0.4720
Epoch 4/50
250/250 - 2s - 7ms/step - loss: 4697959.0000 - r2_score: 0.5170 - val_loss: 4528683.0000 - val_r2_score: 0.5470
Epoch 5/50
250/250 - 2s - 7ms/step - loss: 4156051.0000 - r2_score: 0.5727 - val_loss: 4191121.0000 - val_r2_score: 0.5808
Epoch 6/50
250/250 - 2s - 6ms/step - loss: 3924155.0000 - r2_score: 0.5966 - val_loss: 4043795.2500 - val_r2_score: 0.5955
Epoch 7/50
250/250 - 2s - 7ms/step - loss: 3795661.2500 - r2_score: 0.6098 - val_loss: 3929844.0000 - val_r2_score: 0.6069
Epoch 8/50
250/250 - 2s - 7ms/step - loss: 3680635.5000 - r2_score: 0.6216 - val_loss: 3814656.5000 - val_r2_score: 0.6184
Epo

In [20]:
y_pred = model.predict(X_test_t)

625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [21]:
r2_score(y_test, y_pred)

0.6787251944368842

In [22]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization, Input
from keras.regularizers import l1_l2
from keras.optimizers import Adam, RMSprop, SGD
from keras.callbacks import EarlyStopping

import optuna
import tensorflow as tf


def objective(trial):

    # Hyperparameters
    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-2,
        log=True
    )

    n_layers = trial.suggest_int(
        "n_layers",
        1,
        4
    )

    optimizer_name = trial.suggest_categorical(
        "optimizer",
        ["Adam", "RMSprop", "SGD"]
    )

    activation = trial.suggest_categorical(
        "activation",
        ["relu", "tanh"]
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [32, 64, 128, 256]
    )

    # Model
    model = Sequential()

    model.add(Input(shape=(X_train_t.shape[1],)))

    for i in range(n_layers):

        units = trial.suggest_categorical(
            f'units_{i}',
            [32, 64, 128, 256]
        )

        dropout = trial.suggest_float(
            f'dropout_{i}',
            0.1,
            0.5
        )

        reg = trial.suggest_float(
            f'reg_{i}',
            1e-6,
            1e-3,
            log=True
        )

        model.add(
            Dense(
                units,
                activation=activation,
                kernel_regularizer=l1_l2(l1=reg, l2=reg)
            )
        )

        model.add(BatchNormalization())
        model.add(Dropout(dropout))

    # Output Layer
    model.add(Dense(1, activation='linear'))

    # Optimizer
    opt_map = {
        "Adam": Adam(learning_rate=learning_rate),
        "RMSprop": RMSprop(learning_rate=learning_rate),
        "SGD": SGD(learning_rate=learning_rate)
    }

    model.compile(
        optimizer=opt_map[optimizer_name],
        loss='mse',
        metrics=['mae']
    )

    # Early Stopping
    early = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    # Training
    history = model.fit(
        X_train_t,
        y_train,
        validation_data=(X_val_t, y_val),
        epochs=100,
        batch_size=batch_size,
        callbacks=[early],
        verbose=0
    )

    # Return minimum validation loss
    return min(history.history['val_loss'])

In [23]:
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=21))
study.optimize(objective, n_trials=30, show_progress_bar=True)  

[I 2026-06-29 15:12:55,541] A new study created in memory with name: no-name-47ad43d2-0afc-4877-83fa-440a366c7c9a


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-29 15:28:52,874] Trial 0 finished with value: 3533359.0 and parameters: {'learning_rate': 0.0001251554487151282, 'n_layers': 2, 'optimizer': 'Adam', 'activation': 'tanh', 'batch_size': 32, 'units_0': 32, 'dropout_0': 0.44547985782328947, 'reg_0': 0.0001891609587474211, 'units_1': 32, 'dropout_1': 0.3853441711537915, 'reg_1': 6.486483504947504e-06}. Best is trial 0 with value: 3533359.0.
[I 2026-06-29 15:29:55,043] Trial 1 finished with value: 3092342.25 and parameters: {'learning_rate': 0.005107469054556476, 'n_layers': 4, 'optimizer': 'Adam', 'activation': 'relu', 'batch_size': 128, 'units_0': 128, 'dropout_0': 0.3459298844593739, 'reg_0': 9.083813718648172e-05, 'units_1': 128, 'dropout_1': 0.45933613543387775, 'reg_1': 2.95186624708792e-05, 'units_2': 256, 'dropout_2': 0.37954431558099533, 'reg_2': 2.3076552208868012e-05, 'units_3': 64, 'dropout_3': 0.19248492597613429, 'reg_3': 5.9070859581875536e-05}. Best is trial 1 with value: 3092342.25.
[I 2026-06-29 15:30:53,180] Tr

In [24]:
study.best_params

{'learning_rate': 0.0022920477373681927,
 'n_layers': 4,
 'optimizer': 'Adam',
 'activation': 'relu',
 'batch_size': 128,
 'units_0': 32,
 'dropout_0': 0.17315767766990398,
 'reg_0': 4.112227057552129e-05,
 'units_1': 256,
 'dropout_1': 0.4014166549783429,
 'reg_1': 5.141570801285966e-05,
 'units_2': 32,
 'dropout_2': 0.11813245728931282,
 'reg_2': 1.2880029235205423e-06,
 'units_3': 256,
 'dropout_3': 0.2172814503394815,
 'reg_3': 8.200571639172983e-06}

In [25]:
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, BatchNormalization
from keras.regularizers import l1_l2
from keras.optimizers import Adam

model = Sequential()

# Input Layer
model.add(Input(shape=(X_train_t.shape[1],)))

# Hidden Layer 1
model.add(Dense(
    32,
    activation='relu',
    kernel_initializer='he_normal',
    kernel_regularizer=l1_l2(l1=4.112227057552129e-05,
                             l2=4.112227057552129e-05)
))
model.add(BatchNormalization())
model.add(Dropout(0.17315767766990398))

# Hidden Layer 2
model.add(Dense(
    256,
    activation='relu',
    kernel_initializer='he_normal',
    kernel_regularizer=l1_l2(l1=5.141570801285966e-05,
                             l2=5.141570801285966e-05)
))
model.add(BatchNormalization())
model.add(Dropout(0.4014166549783429))

# Hidden Layer 3
model.add(Dense(
    32,
    activation='relu',
    kernel_initializer='he_normal',
    kernel_regularizer=l1_l2(l1=1.2880029235205423e-06,
                             l2=1.2880029235205423e-06)
))
model.add(BatchNormalization())
model.add(Dropout(0.11813245728931282))

# Hidden Layer 4
model.add(Dense(
    256,
    activation='relu',
    kernel_initializer='he_normal',
    kernel_regularizer=l1_l2(l1=8.200571639172983e-06,
                             l2=8.200571639172983e-06)
))
model.add(BatchNormalization())
model.add(Dropout(0.2172814503394815))

# Output Layer (Regression)
model.add(Dense(1, activation='linear'))

In [26]:
from keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0022920477373681927),
    loss='mse',
    metrics=['mae']
)

In [27]:
from keras.callbacks import EarlyStopping

es = EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=10,
    restore_best_weights=True
)

In [28]:
history = model.fit(
    X_train_t,
    y_train,
    epochs=100,
    batch_size=128,
    validation_data=(X_val_t, y_val),
    callbacks=[es],
    verbose=2
)

Epoch 1/100
500/500 - 7s - 13ms/step - loss: 15895433.0000 - mae: 2780.8108 - val_loss: 11625768.0000 - val_mae: 2437.2644
Epoch 2/100
500/500 - 3s - 6ms/step - loss: 7467625.0000 - mae: 1752.8638 - val_loss: 3616161.7500 - val_mae: 1013.0867
Epoch 3/100
500/500 - 3s - 6ms/step - loss: 4073930.0000 - mae: 1138.1952 - val_loss: 3125834.2500 - val_mae: 963.6750
Epoch 4/100
500/500 - 3s - 6ms/step - loss: 3662647.0000 - mae: 1077.3848 - val_loss: 3103184.5000 - val_mae: 972.6416
Epoch 5/100
500/500 - 3s - 7ms/step - loss: 3571523.5000 - mae: 1082.6952 - val_loss: 3080447.0000 - val_mae: 986.2150
Epoch 6/100
500/500 - 3s - 6ms/step - loss: 3554421.7500 - mae: 1075.4309 - val_loss: 3190945.5000 - val_mae: 1016.6940
Epoch 7/100
500/500 - 3s - 6ms/step - loss: 3509849.7500 - mae: 1074.5709 - val_loss: 3136967.2500 - val_mae: 962.0893
Epoch 8/100
500/500 - 3s - 6ms/step - loss: 3485486.0000 - mae: 1065.2423 - val_loss: 3162424.7500 - val_mae: 990.0703
Epoch 9/100
500/500 - 3s - 6ms/step - loss

In [29]:
y_pred = model.predict(X_test_t).flatten()

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 945us/step  


In [30]:
r2_score(y_test,y_pred)

0.6797186843735517

In [31]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("MAE :", mean_absolute_error(y_test, y_pred))
print("MSE :", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2  :", r2_score(y_test, y_pred))

MAE : 978.3998083247681
MSE : 3132277.0983333304
RMSE: 1769.8240303299451
R2  : 0.6797186843735517
